In [ ]:
import subprocess
import sys
import json
import pandas as pd

# Log

In [ ]:
report_df = pd.DataFrame(columns=['method','stage','user','score','widePortion','transitionPortion','transitionCount','head_qu_distance','head_qu_mean_velocity','head_qu_mean_acc','rightHand_qu_distance','rightHand_qu_mean_velocity','rightHand_qu_mean_acc','leftHand_qu_distance','leftHand_qu_mean_velocity','leftHand_qu_mean_acc','player_mean_velocity','player_mean_acc'])

In [ ]:
userList = [
  'P002',
  'P004',
  'P009',
  'P011',
  'P012',
  'P013',
  'P026',
  'P027',
  'P028',
  'P029',
  'P032',
  'P033',
  'P035',
]

methodList = [
  'city_Dynamic_IdleDetection',
  'maze_Dynamic_IdleDetection',
  'city_Dynamic_TargetDetection',
  'shooting_Dynamic_TargetDetection',
  'puzzle_Dynamic_TargetDetection',
  'shooting_Dynamic_Manual',
  'maze_Dynamic_Manual',
  'puzzle_Dynamic_HandPosition',
  'city_Static',
  'puzzle_Static',
  'shooting_Static',
  'maze_Static'
]

In [ ]:
report_script = "report_formal.py"

for i in range(len(userList)):
  for j in range(len(methodList)):
    arg = "../data/chi24/formal/" + userList[i] + "/LOG_Formal_" + userList[i] + "_" + methodList[j] + "_0.txt"

    completed_process = subprocess.run(
      [sys.executable, report_script, "--file", arg],
      check=True,
      capture_output=True,
      text=True,
    )
    output = completed_process.stdout

    # print error
    print(completed_process.stderr)

    print(output.split("\n")[-2].replace("'", "\""))
    json_result = output.split("\n")[-2].replace("'", "\"") 
    result = json.loads(json_result)
    report_df = pd.concat([report_df, pd.DataFrame([result])], ignore_index=True)

In [ ]:
report_df

# Questionnaire

In [ ]:
data = [] # your list with json objects (dicts)

with open('../data/chi24/response_formal.json', 'r', encoding='utf-8') as json_file:
  data = json.load(json_file)

print(data[13])

In [ ]:
questionnaires = []

for i, k in enumerate(data):
  quest = {
    'method': 'na',
    'user': 'na',
    'pref': -1,
    'imms': -1,
    'tlx': -1,
    'ssq': -1,
    'nausea': -1,
    'oculomotor': -1,
    'disorientation': -1
  }

  quest['method'] = k['trigger'][0]
  quest['user'] = k['user'][0]
  quest['pref'] = int(k['answers'][3]['answer'][0])
  quest['imms'] = int(k['answers'][4]['answer'][0])

  # tlx
  md = int(k['answers'][5]['answer'][0]) * 5/2 + 5/2 # mental demand
  ps = int(k['answers'][6]['answer'][0]) * 5/2 + 5/2 # physical demand
  td = int(k['answers'][7]['answer'][0]) * 5/2 + 5/2 # temporal demand
  pe = int(k['answers'][8]['answer'][0]) * 5/2 + 5/2 # performance
  ef = int(k['answers'][9]['answer'][0]) * 5/2 + 5/2 # effort
  fr = int(k['answers'][10]['answer'][0]) * 5/2 + 5/2 # frustration

  weights = k['answers'][11]['answer']
  tlx = 5*(md * weights[0] + ps * weights[1] + td * weights[5] + pe * weights[3] + ef * weights[4] + fr * weights[2])/15
  quest['tlx'] = tlx

  # ssq
  naus_index = [12, 17, 18, 19, 20, 26, 27]
  ocul_index = [12, 13, 14, 15, 16, 20, 22]
  diso_index = [16, 19, 21, 22, 23, 24, 25]
  
  ssq = 0
  naus = 0
  ocul = 0
  diso = 0

  for j in naus_index:
    naus += (int(k['answers'][j]['answer'][0])-1)/2
  for j in ocul_index:
    ocul += (int(k['answers'][j]['answer'][0])-1)/2
  for j in diso_index:
    diso += (int(k['answers'][j]['answer'][0])-1)/2
  
  ssq = (naus + ocul + diso)*3.74
  quest['ssq'] = ssq

  naus = naus*9.54
  ocul = ocul*7.58
  diso = diso*13.92

  quest['nausea'] = naus
  quest['oculomotor'] = ocul
  quest['disorientation'] = diso

  questionnaires.append(quest)

In [ ]:
questionnaires_df = pd.DataFrame(questionnaires)
display(questionnaires_df)

# Combine

In [ ]:
# rename user name in questionnaires_df
for i in range(questionnaires_df.shape[0]):
  questionnaires_df.loc[i, 'user'] = replace_name(questionnaires_df.loc[i, 'user'])

In [ ]:
combine_df = pd.merge(report_df, questionnaires_df, how='left', on=['method', 'user'])

In [ ]:
display(combine_df)

In [ ]:
# export to csv
combine_df.to_csv('./combine_formal.csv', index=False)